# ResNet-50 a1_in1k — DIMER image classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/resnet50-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/resnet50-classification-pipeline/blob/main/tutorials/resnet50_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Fresnet50.a1__in1k-ffcc4d?style=flat)](https://huggingface.co/timm/resnet50.a1_in1k) [![Upstream](https://img.shields.io/badge/Upstream-huggingface%2Fpytorch--image--models-181717?style=flat&logo=github&logoColor=white)](https://github.com/huggingface/pytorch-image-models) [![arXiv](https://img.shields.io/badge/arXiv-2110.00476-b31b1b.svg)](https://arxiv.org/abs/2110.00476)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** ImageNet-1k single-label image classification (1000 classes) and in-kernel fine-tuning on custom classes (full or head-only) using the pinned `timm/resnet50.a1_in1k` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/resnet50_classification_pipeline/pipeline.py` at revision `3322040e1d16`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `767268603ca0cb0bfe326fa87277f19c419566ef` (~103 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (the kernel's own packages are left alone, so no restart is needed), stages and digest-verifies the pinned snapshot, generates the deterministic synthetic sample image, validates it into an input manifest, classifies it with the pretrained ImageNet-1k head, writes the zero-shot evaluation report, downloads and digest-verifies the `Cleanlab/cifar-10-subset` tutorial dataset and validates a seeded, pair-grouped split of 100 images per class (160 training, 40 held out; both copies of a photo stay on one side), fine-tunes a new two-class head on the frozen pretrained backbone (**head-only fine-tuning**, one epoch, AdamW and cross-entropy), exports the artifact (`model.safetensors` + `model-config.json`), reloads it across a fresh boundary and checks it reproduces the in-memory model, evaluates it on the 40 held-out images with a 95 % interval against the majority-class baseline, and writes outputs and provenance. No repository clone, DIMER worker or service, credential, upload dialog or configuration edit is required (NOTEBOOK_SPEC 2.2 §5). Measured times: the previous notebook version (in-kernel install, a 32-image fine-tuning set) passed on a Kaggle Tesla T4 in 182.3 s including its install (2026-09-14); this version's default path took 17.5 s of cell time in a local Windows CPU check (2026-10-09, files pre-staged; 4.8 s of it the head-only fine-tuning on 160 images), and the optional full fine-tuning of Section 12 took 31 s there. Building the isolated environment (PyTorch with its CUDA libraries) and downloading the 103 MB checkpoint come on top of these and usually take a few minutes (an estimate; no run of this version is recorded yet).

**Bring Your Own Data:** Two optional branches, both off by default and never part of the default path. `USE_BYOD = True` in Section 4 takes one image (upload dialog on Colab, or a file path in `BYOD_IMAGE_PATH` on any runtime) through the same validation, classification, evaluation-report and export cells as the synthetic sample. `USE_BYOD_DATASET = True` in Section 8 takes a `.zip` of class folders (upload, or `BYOD_DATASET_PATH`) through the same validation, seeded split, fine-tuning, export, fresh-reload and held-out evaluation cells as the tutorial dataset; Section 8 refuses a bad archive (named file or class, and the rule) before any training. After the default run, set the field, select that section's cell and choose **Runtime → Run after**. The archive contract, the limits and the privacy guidance are in the Prerequisites; uploads stay inside this runtime.

At inference the network maps one normalized 3×224×224 tensor to 1000 logits in a single forward pass; the pipeline applies a softmax and reports the argmax class plus the top-k classes with their scores. **In-kernel fine-tuning:** this notebook demonstrates both zero-shot base inference on ImageNet-1k classes and fine-tuning on custom classes with PyTorch AdamW and cross-entropy loss. `fit` replaces the 1000-class head with a new linear classifier sized to your classes and, by default, trains **only that classifier** with the pretrained backbone frozen (head-only fine-tuning: 4,098 parameters; the backbone's BatchNorm statistics stay frozen too); with `TRAINABLE = 'all'` it trains every parameter, the backbone included (full fine-tuning: 23.5 M parameters). The notebook prints the trainable and frozen parameter counts before training. The carried pipeline module adds snapshot verification, input and dataset validation, `fit`, a held-out evaluation report with its uncertainty, and the `top_k_accuracy`, `validate_inputs` and `evaluation_report` helpers. The default inference sample is a synthetic image generated in code; its prediction is demonstration (plumbing) evidence, not a production-quality or benchmark claim. The fine-tuning result is measured on **40 held-out images** (20 photos, each in two copies), so it is a tutorial metric with a wide 95 % interval, not evidence that the method works in general.

**Who this is for.** A learner who knows basic Python and has met the idea of an image classifier, and wants to see how a pretrained network is used, adapted to new classes and checked honestly. No prior experience with ResNet, timm or fine-tuning is assumed; each term is explained where it is first used and again in the **Glossary** at the end. No GPU is required (CPU works; a GPU is used automatically).

**Input → Model → Output.**

| | ImageNet-1k inference (Sections 4–7) | Fine-tuning (Sections 8–11) |
|---|---|---|
| Input | one RGB image (any size up to 4096 px), resized to 235 px and center-cropped to 224×224 | a labelled image set: class folders, split into training and held-out images |
| Model | ResNet-50 backbone + the pretrained 1000-class head | the same backbone + a new head sized to your classes, trained with AdamW |
| Output | the argmax class and a ranked top-5 with softmax scores (not probabilities of being right) | an artifact (`model.safetensors` + `model-config.json`) and held-out accuracy with a 95 % interval and the majority-class baseline |

**How to use this notebook.** Choose a runtime (Colab, Kaggle or Linux Jupyter; a GPU is faster, CPU works), then **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried code and the model verification — and can be run without study; their code is collapsed. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the default path. Each learner section states what it does and asks you to **predict** before it runs; the next section opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer. Each optional experiment names the field to change and the cells to re-run. Section 12 is a **change-one-thing activity**. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional.

**Roadmap:** *core concepts* — 4 the sample → 5 validate it → 6 classify with the pretrained model → 7 the evaluation report; *evaluation practice* — 8 the fine-tuning data and its split → 9 fine-tune → 10 reload, check equivalence and evaluate on held-out images → 11 export outputs and provenance (*engineering*) → 12 **change one thing: train every layer** → conclude.

**Learning objectives:** by the end you should be able to (1) explain *image → resize and crop → ResNet-50 → softmax → argmax and top-5* and say why the score is not a probability of being right (Section 6); (2) explain why a prediction on the synthetic gradient is `not-measurable` and what would make it measurable (Section 7); (3) identify how many parameters head-only and full fine-tuning train, from the printed counts (Sections 9 and 12); (4) interpret a held-out accuracy together with its sample size, its 95 % interval and the majority-class baseline, and decide whether it supports a claim (Section 10); (5) check that an exported artifact reproduces the evaluated model (Section 10); and (6) predict, measure and explain how full fine-tuning compares with head-only fine-tuning on the same split, including why it can do worse (Section 12).

**This notebook does not demonstrate:** object detection, segmentation, multi-label tagging, OCR, open-vocabulary classification, feature/embedding extraction. The base label space is fixed to the 1000 ImageNet-1k classes; an image whose subject is outside that space still receives a label unless adapted via in-kernel fine-tuning. The fine-tuning sections demonstrate the procedure on a tiny sample; they do not establish that fine-tuning improves accuracy on any task.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; what an image classifier does. The notebook explains the softmax over class logits (and why it is not a calibrated probability), fine-tuning (full vs head-only), AdamW, cross-entropy, a held-out split, the majority-class baseline and a confidence interval where they are first used; the Glossary repeats them.
- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab, Kaggle or Linux Jupyter. Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter, and a Windows or macOS kernel is not supported (Section 1 stops with that message). The default path runs on CPU and uses CUDA automatically when available; inference and training are float32. The locked install (PyTorch 2.14.0 with its CUDA libraries) is the largest download. Time: the previous notebook version passed on a Kaggle Tesla T4 in 182.3 s including its in-kernel install (2026-09-14); this version's cells took 17.5 s in a local Windows CPU check with the files pre-staged (2026-10-09); the isolated install and the downloads (103 MB checkpoint) add a few minutes (an estimate).
- **Data (default path):** the inference sample is a deterministic 256×256 RGB gradient generated in code (no ground truth). The fine-tuning sections download one dataset: [`Cleanlab/cifar-10-subset`](https://huggingface.co/datasets/Cleanlab/cifar-10-subset) at commit `bb5a7aab` (MIT licence), a 986,707-byte `.zip` from `huggingface.co`, refused unless its SHA-256 is `66f90a4f…`. It holds 400 CIFAR-10 images of 32×32 px (upscaled to 224 px by the model's preprocessing) in two classes, `frog` and `truck`, each stored twice: an original and a darkened copy. The notebook keeps 100 per class — 50 photos, each with both copies — and holds out 20 per class (10 photos), always keeping a photo's two copies on the same side of the split. If the download fails, Section 8 falls back to a synthetic two-class stripes dataset and says so.
- **BYOD image (Section 4):** one image file decodable by Pillow (PNG, JPEG, WebP, BMP and similar), any colour mode, longest side at most 4096 px; optionally its ImageNet-1k class index (0–999).
- **BYOD dataset (Section 8):** one `.zip` of at most 200 MB and 2,000 images (`.png`, `.jpg`, `.jpeg`, `.webp`, `.bmp`; longest side at most 4096 px) in **either** `<class>/<image>` folders (split here: per class, seeded, 20 % held out, at least 2 images per class) **or** `train/<class>/…` plus `val/<class>/…` (also `valid/`, `validation/`) used as given. A `test/` folder is refused (it would otherwise be pooled into training; keep your test set for a final check outside this notebook, or rename it to `val/`). At least 2 and at most 100 classes; every image must be inside a class folder; every validation class must also exist in `train/` and every training class in `val/`; other files are skipped and listed. Section 8 checks all of this and decodes every image before any training, and names the file or class it refuses. Validation is structural, not semantic: nothing checks that an image shows its folder's class.
- **Privacy:** Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `timm/resnet50.a1_in1k` snapshot (~103 MB in total) at revision `767268603ca0…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 45 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 45 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchaudio==2.11.0',
    'torchvision==0.29.0',
    'timm==1.0.29',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '01ad46e1e700bf2ca05e683a074781b836df6e614673501a1ac087d6426c37b3'
LOCKED_PACKAGES = 45
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   huggingface-hub
    #   torch
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   resnet50-classification-pipeline (pyproject.toml)
    #   timm
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   resnet50-classification-pipeline (pyproject.toml)
    #   torchvision
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via huggingface-hub
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   resnet50-classification-pipeline (pyproject.toml)
    #   torchvision
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   timm
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via huggingface-hub
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   resnet50-classification-pipeline (pyproject.toml)
    #   timm
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
timm==1.0.29 \
    --hash=sha256:1af8d12bb7e15c5f96e98d60b7b8319cd7f31730778bf1af58e912a7ce171576 \
    --hash=sha256:4266486e355c9beb9432e689de580d09e2efe7fa92a9a79e76c03e7763eac724
    # via resnet50-classification-pipeline (pyproject.toml)
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   resnet50-classification-pipeline (pyproject.toml)
    #   timm
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via resnet50-classification-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via
    #   resnet50-classification-pipeline (pyproject.toml)
    #   timm
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via huggingface-hub
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 45, 'setup_seconds': 50}


The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 2609).


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `timm` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchaudio==2.11.0',
    'torchvision==0.29.0',
    'timm==1.0.29',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'resnet50-classification-pipeline',
    'repository_revision': '3322040e1d16c1ab27babf31bb91fffa7f007a3a',
    'embedded_module': 'src/resnet50_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/resnet50_classification_pipeline/pipeline.py'],
    'module_sha256': '993f39271ae0e389e6eaa193f8cde21c201d820cdc147c21bf3dee31a895b16f',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'resnet50-classification-pipeline', 'repository_revision': '3322040e1d16c1ab27babf31bb91fffa7f007a3a', 'embedded_module': 'src/resnet50_classification_pipeline/pipeline.py', 'embedded_modules': ['src/resnet50_classification_pipeline/pipeline.py'], 'module_sha256': '993f39271ae0e389e6eaa193f8cde21c201d820cdc147c21bf3dee31a895b16f', 'generator': 'build_notebook.py/2.1', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'torch': '2.14.0+cu130', 'timm': '1.0.29', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/resnet50_classification_pipeline/` @ `3322040e1d16`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/resnet50_classification_pipeline/pipeline.py`

In [ ]:
# @title Infrastructure: carried module `src/resnet50_classification_pipeline/pipeline.py` (you may run it without reading it)
"""ImageNet-1k classification with the pinned ``timm/resnet50.a1_in1k`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or,
when explicitly allowed, from the Hugging Face Hub at the pinned revision. Preprocessing is
the upstream ``pretrained_cfg`` (resize/crop/normalize) resolved through ``timm.data``.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image, UnidentifiedImageError

MODEL_ID = "timm/resnet50.a1_in1k"
MODEL_REVISION = "767268603ca0cb0bfe326fa87277f19c419566ef"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "resnet50-a1"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

NUM_CLASSES = 1000
MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MAX_BATCH = 64  # images per predict() call
DEFAULT_TOP_K = 5
DECISION_RULE = "argmax"  # the label reported as `predicted_index` is the softmax argmax; no threshold


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _hub_reference(model_id: str, revision: str) -> str:
    """timm's ``hf-hub:owner/name@revision`` form; ``hf_split`` passes ``revision=`` to hf_hub_download."""
    return f"hf-hub:{model_id}@{revision}"


def top_k_accuracy(predictions: Sequence[Any], targets: Sequence[int], k: int = 1) -> float:
    """Fraction of items whose target index is among the first ``k`` predicted indices.

    ``predictions`` may be the per-image dicts returned by ``predict`` or plain index sequences.
    """
    if len(predictions) != len(targets):
        raise ValueError("predictions and targets must have the same length")
    if not predictions:
        raise ValueError("predictions must not be empty")
    if not isinstance(k, int) or k < 1:
        raise ValueError("k must be a positive integer")
    hits = 0
    for pred, target in zip(predictions, targets, strict=True):
        ranked = pred["top_k"] if isinstance(pred, Mapping) else pred
        indices = [int(item["index"]) if isinstance(item, Mapping) else int(item) for item in ranked]
        hits += int(target in indices[:k])
    return hits / len(predictions)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image or a sequence of them; any mode, converted to RGB",
    "image_side_px": [1, MAX_IMAGE_SIDE],
    "batch": [1, MAX_BATCH],
    "top_k": [1, NUM_CLASSES],
    "preprocessing": "resize to 235 px, center-crop 224x224 (crop_pct 0.95, bicubic), ImageNet normalisation",
}


def _check_inputs(images: Any, top_k: int, max_classes: int = NUM_CLASSES) -> list[Image.Image]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the images as a list."""
    if isinstance(images, Image.Image):
        images = [images]
    if not isinstance(images, Sequence) or isinstance(images, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if not 1 <= len(images) <= MAX_BATCH:
        raise ValueError(f"batch size must be between 1 and MAX_BATCH={MAX_BATCH}, got {len(images)}")
    for image in images:
        if not isinstance(image, Image.Image):
            raise TypeError(f"each image must be a PIL.Image.Image, got {type(image).__name__}")
        width, height = image.size
        if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
            raise ValueError(f"image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= max_classes:
        raise ValueError(f"top_k must be between 1 and {max_classes}")
    return list(images)


def validate_inputs(
    images: Image.Image | Sequence[Image.Image],
    top_k: int = DEFAULT_TOP_K,
    *,
    names: Sequence[str] | None = None,
    num_classes: int = NUM_CLASSES,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``predict`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    checked = _check_inputs(images, top_k, max_classes=num_classes)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)}
            for i, image in enumerate(checked)
        ],
        "top_k": top_k,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], targets: Sequence[int] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``targets`` (one ImageNet-1k index per prediction) the report carries ``top_k_accuracy``
    at k=1 and k=5 as sample-sanity evidence; without them the verdict is ``not-measurable`` and
    the report says what labelled data would make the task measurable.
    """
    predictions = result["predictions"]
    base = {
        "task": "imagenet-1k single-label classification",
        "decision_rule": result.get("decision_rule", DECISION_RULE),
        "sample_kind": sample_kind,
        "n_predictions": len(predictions),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if targets is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth class index was supplied for the evaluated images",
            "needs": (
                "labelled photographs with ImageNet-1k class indices (0-999), e.g. a held-out sample of your "
                "own data, scored with top_k_accuracy against the majority-class baseline of that sample"
            ),
        }
    top_k = int(result.get("top_k", DEFAULT_TOP_K))
    ks = sorted({1, min(5, top_k)})
    return {
        **base,
        "metrics": [
            {
                "id": "top_k_accuracy",
                "k": k,
                "value": top_k_accuracy(predictions, list(targets), k=k),
                "estimation": "single sample, no dispersion estimate",
            }
            for k in ks
        ],
        "verdict": "sample-sanity",
        "reason": f"{len(predictions)} labelled image(s) from the tutorial sample; not a benchmark",
        "needs": "a labelled evaluation set from the deployment domain for any generalisable accuracy claim",
    }


# --- Fine-tuning helpers: trainable set, held-out evaluation with its uncertainty, dataset intake ---

MAX_DATASET_ARCHIVE_BYTES = 200_000_000  # a BYOD .zip larger than this is refused before it is opened
MAX_DATASET_IMAGES = 2000  # image files per dataset archive
MAX_DATASET_CLASSES = 100
IMAGE_SUFFIXES = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
SPLIT_DIR_NAMES = {"train": "train", "val": "val", "valid": "val", "validation": "val"}
# RN-M5: a user's test split is never pooled into training; an archive holding one is refused before decoding.
TEST_DIR_NAMES = {"test", "testing"}


def _check_training_config(epochs: Any, batch_size: Any, learning_rate: Any, weight_decay: Any) -> None:
    """Raise ValueError naming the first training setting outside its accepted range."""
    for name, value in (("epochs", epochs), ("batch_size", batch_size)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f"{name} must be a positive integer, got {value!r}")
    if isinstance(learning_rate, bool) or not isinstance(learning_rate, int | float) or not learning_rate > 0:
        raise ValueError(f"learning_rate must be a positive number, got {learning_rate!r}")
    if isinstance(weight_decay, bool) or not isinstance(weight_decay, int | float) or weight_decay < 0:
        raise ValueError(f"weight_decay must be a number >= 0, got {weight_decay!r}")


def set_trainable(model: Any, *, train_backbone: bool) -> dict[str, Any]:
    """Apply the fine-tuning rule to a timm classifier and count what it trains.

    ``train_backbone=True`` leaves every parameter trainable (full fine-tuning); ``False`` freezes every
    parameter except those of ``model.get_classifier()`` (head-only fine-tuning).
    """
    if not isinstance(train_backbone, bool):
        raise TypeError("train_backbone must be True (full fine-tuning) or False (head only)")
    if not train_backbone:
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        for parameter in model.get_classifier().parameters():
            parameter.requires_grad_(True)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {
        "method": "full fine-tuning" if train_backbone else "head-only fine-tuning",
        "trainable": "all" if train_backbone else "head",
        "trainable_parameters": trainable,
        "frozen_parameters": total - trainable,
    }


def trainable_parameter_counts(num_classes: int, *, train_backbone: bool) -> dict[str, Any]:
    """The counts ``fit`` will train, computed on the bare architecture (no weights are read)."""
    import timm

    model = timm.create_model(MODEL_ID.split("/", 1)[1], pretrained=False, num_classes=num_classes)
    return set_trainable(model, train_backbone=train_backbone)


def wilson_interval(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion (95 % with the default ``z``)."""
    if n < 1 or not 0 <= successes <= n:
        raise ValueError(f"need 0 <= successes <= n and n >= 1, got {successes}/{n}")
    p = successes / n
    denominator = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denominator
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / denominator
    low = 0.0 if successes == 0 else max(0.0, centre - half)
    high = 1.0 if successes == n else min(1.0, centre + half)
    return low, high


def finetune_evaluation_report(
    true_indices: Sequence[int],
    predicted_indices: Sequence[int],
    class_names: Sequence[str],
    *,
    item_ids: Sequence[str] | None = None,
    scores: Sequence[float] | None = None,
    sample_kind: str = "tutorial",
) -> dict[str, Any]:
    """Held-out evaluation of a fine-tuned classifier, reported with its uncertainty.

    The verdict is ``sample-sanity`` (never a success claim). ``comparison_to_baseline`` is read from the 95 %
    Wilson interval of the accuracy against the majority-class baseline of the same items: ``above-baseline``
    only when the whole interval lies above the baseline, ``below-baseline`` when it lies below, otherwise
    ``indistinguishable-from-baseline``.
    """
    n = len(true_indices)
    if n < 1 or len(predicted_indices) != n:
        raise ValueError("true_indices and predicted_indices must be non-empty and of equal length")
    ids = list(item_ids) if item_ids is not None else [f"item-{i}" for i in range(n)]
    correct = sum(int(t == p) for t, p in zip(true_indices, predicted_indices, strict=True))
    accuracy = correct / n
    counts = {name: sum(1 for t in true_indices if t == i) for i, name in enumerate(class_names)}
    majority_class = max(counts, key=lambda name: (counts[name], -list(class_names).index(name)))
    baseline = counts[majority_class] / n
    low, high = wilson_interval(correct, n)
    if low > baseline:
        comparison = "above-baseline"
    elif high < baseline:
        comparison = "below-baseline"
    else:
        comparison = "indistinguishable-from-baseline"
    per_class = {
        name: {
            "total": counts[name],
            "correct": sum(
                1 for t, p in zip(true_indices, predicted_indices, strict=True) if t == i and p == i
            ),
        }
        for i, name in enumerate(class_names)
    }
    misclassified = [
        {
            "id": ids[k],
            "true_label": class_names[t],
            "predicted_label": class_names[p],
            **({"score": float(scores[k])} if scores is not None else {}),
        }
        for k, (t, p) in enumerate(zip(true_indices, predicted_indices, strict=True))
        if t != p
    ]
    return {
        "task": "single-label classification fine-tuning evaluation",
        "verdict": "sample-sanity",
        "sample_kind": sample_kind,
        "n": n,
        "correct": correct,
        "classes": list(class_names),
        "metrics": {
            "accuracy": accuracy,
            "accuracy_wilson_95": [low, high],
            "majority_baseline_accuracy": baseline,
            "majority_baseline_class": majority_class,
            "accuracy_minus_baseline": accuracy - baseline,
        },
        "comparison_to_baseline": comparison,
        "per_class": per_class,
        "misclassified": misclassified,
        "estimation": (
            f"tutorial metric on {n} held-out image(s), not a benchmark: one seeded split, one training run, "
            "95 % Wilson score interval for the accuracy; one image moves the accuracy by "
            f"{100 / n:.1f} percentage points"
        ),
        "needs": "a larger labelled held-out set from the deployment domain for any accuracy claim",
    }


def synthetic_stripes_dataset(side: int = 64, per_class: int = 6, n_val: int = 2) -> dict[str, Any]:
    """Deterministic two-class fallback (horizontal vs vertical stripes) for a runtime without the dataset."""
    import numpy as np

    if not 1 <= n_val < per_class:
        raise ValueError("need 1 <= n_val < per_class")
    classes = ["synthetic_horizontal_stripe", "synthetic_vertical_stripe"]
    keys = ("train_images", "train_targets", "val_images", "val_targets")
    split: dict[str, list[Any]] = {key: [] for key in keys}
    ids: dict[str, list[str]] = {"train": [], "val": []}
    digest = hashlib.sha256()
    for cls_idx, pattern in enumerate(("horizontal", "vertical")):
        for i in range(per_class):
            arr = np.zeros((side, side, 3), dtype=np.uint8)
            if pattern == "horizontal":
                arr[::16, :, 0] = 255
            else:
                arr[:, ::16, 1] = 255
            arr[:, :, 2] = (i * 30) % 255
            digest.update(arr.tobytes())
            part = "train" if i < per_class - n_val else "val"
            split[f"{part}_images"].append(Image.fromarray(arr))
            split[f"{part}_targets"].append(cls_idx)
            ids[part].append(f"{pattern}_{i}")
    return {
        **split,
        "classes": classes,
        "train_ids": ids["train"],
        "val_ids": ids["val"],
        "manifest": {
            "source": "synthetic stripes fallback (generated in code)",
            "sha256": digest.hexdigest(),
            "layout": "generated",
            "classes": classes,
            "per_class": {c: {"train": per_class - n_val, "val": n_val} for c in classes},
            "skipped": [],
        },
    }


def _decode_member(archive: Any, member: str) -> Image.Image:
    """Open one archive member, check its size against MAX_IMAGE_SIDE before decoding, then decode to RGB."""
    try:
        image = Image.open(io.BytesIO(archive.read(member)))
        validate_inputs(image, top_k=1, names=[member], num_classes=1)
        return image.convert("RGB")
    except (UnidentifiedImageError, Image.DecompressionBombError, OSError) as exc:
        raise ValueError(
            f"{member}: not a decodable image ({type(exc).__name__}); PNG/JPEG/WebP/BMP expected. "
            "Remove or replace it and upload again"
        ) from None
    except ValueError as exc:
        raise ValueError(f"{member}: {exc}; resize or remove it and upload again") from None


def load_image_zip(
    zip_bytes: bytes,
    *,
    seed: int,
    validation_split: float = 0.2,
    subset_per_class: int | None = None,
    source: str = "uploaded archive",
) -> dict[str, Any]:
    """Read a class-folder image archive into a seeded train/val split, refusing bad layouts before training.

    Two layouts are accepted: ``<class>/<image>`` folders, split here, or ``train/<class>/...`` plus
    ``val/<class>/...`` (``valid``/``validation`` also accepted) used as given.

    The split here is **pair-grouped** and seeded, per class. Images of one class whose file names
    (the last path component) are equal form one group, e.g. ``original_images/frog/image_7.png`` and
    ``darkened_images/frog/image_7.png``; a group always lands on one side of the split. Groups are
    listed in archive order of their first image and shuffled with one ``random.Random(seed)`` shared
    across classes (classes in archive order). With ``subset_per_class`` the leading groups are kept
    while their image total stays within ``subset_per_class``. The first
    ``max(1, int(n_groups * validation_split))`` kept groups are validation, the rest training. When no
    two images share a name this equals the previous per-image split. Every image must sit
    inside a class folder, every validation class must exist in ``train`` and every train class in
    ``val``, a ``test/`` folder is refused (never pooled into training), at least 2 classes are needed,
    every image is decoded and checked against ``MAX_IMAGE_SIDE`` here, and the archive limits are
    ``MAX_DATASET_ARCHIVE_BYTES``, ``MAX_DATASET_IMAGES`` and ``MAX_DATASET_CLASSES``. Each refusal names the
    file or class and the rule.
    """
    if len(zip_bytes) > MAX_DATASET_ARCHIVE_BYTES:
        raise ValueError(
            f"archive is {len(zip_bytes)} bytes; limit MAX_DATASET_ARCHIVE_BYTES={MAX_DATASET_ARCHIVE_BYTES}"
        )
    if not 0 < validation_split < 1:
        raise ValueError(f"validation_split must be between 0 and 1, got {validation_split}")
    try:
        archive = zipfile.ZipFile(io.BytesIO(zip_bytes))
    except zipfile.BadZipFile:
        raise ValueError(f"{source}: not a readable .zip archive") from None
    with archive:
        skipped: list[dict[str, str]] = []
        names: list[str] = []
        for info in archive.infolist():
            name = info.filename
            unsafe = ".." in name.replace("\\", "/").split("/") or name.startswith(("/", "\\"))
            if unsafe or ":" in name.split("/")[0]:
                raise ValueError(f"{name}: illegal path (absolute or '..'); refusing the archive")
            if info.is_dir():
                continue
            if name.startswith("__MACOSX") or name.rsplit("/", 1)[-1].startswith("."):
                skipped.append({"file": name, "reason": "system metadata"})
            elif name.lower().endswith(IMAGE_SUFFIXES):
                names.append(name)
            else:
                skipped.append({"file": name, "reason": f"not an image ({', '.join(IMAGE_SUFFIXES)})"})
        if not names:
            raise ValueError(f"{source}: no image files ({', '.join(IMAGE_SUFFIXES)}) found")
        if len(names) > MAX_DATASET_IMAGES:
            raise ValueError(f"{source}: {len(names)} images; limit MAX_DATASET_IMAGES={MAX_DATASET_IMAGES}")

        def parts_of(name: str) -> list[str]:
            return name.strip("/").split("/")

        for name in names:
            if any(p.lower() in TEST_DIR_NAMES for p in parts_of(name)[:-1]):
                raise ValueError(
                    f"{name}: the archive has a test/ folder. This notebook holds out its own validation "
                    "split and would otherwise pool your test images into training; remove test/ (keep it "
                    "for a final check outside this notebook) or rename it to val/"
                )

        def split_of(name: str) -> str | None:
            found = {SPLIT_DIR_NAMES[p.lower()] for p in parts_of(name)[:-1] if p.lower() in SPLIT_DIR_NAMES}
            return found.pop() if len(found) == 1 else ("ambiguous" if found else None)

        def class_of(name: str) -> str:
            parts = parts_of(name)
            if len(parts) < 2 or parts[-2].lower() in SPLIT_DIR_NAMES:
                raise ValueError(
                    f"{name}: image is not inside a class folder; put every image in <class>/ "
                    "(or train/<class>/ and val/<class>/)"
                )
            return parts[-2]

        splits = {name: split_of(name) for name in names}
        if any(s == "ambiguous" for s in splits.values()):
            bad = next(n for n, s in splits.items() if s == "ambiguous")
            raise ValueError(f"{bad}: path names both a train and a val folder")
        presplit = "train" in splits.values() and "val" in splits.values()
        if presplit and any(s is None for s in splits.values()):
            bad = next(n for n, s in splits.items() if s is None)
            raise ValueError(f"{bad}: the archive has train/ and val/ folders, so every image must be in one")

        files: dict[str, dict[str, list[str]]] = {}
        for name in names:
            part = splits[name] if presplit else "all"
            files.setdefault(class_of(name), {}).setdefault(part, []).append(name)
        if presplit:
            classes = sorted(c for c, parts in files.items() if parts.get("train"))
            missing = sorted(c for c, parts in files.items() if not parts.get("train"))
            if missing:
                raise ValueError(
                    f"class(es) {missing} appear in val/ but not in train/; each val class needs train images"
                )
        else:
            classes = sorted(files)
        if len(classes) < 2:
            raise ValueError(f"classification needs at least 2 classes, found {len(classes)}: {classes}")
        if len(classes) > MAX_DATASET_CLASSES:
            raise ValueError(f"{len(classes)} classes; limit MAX_DATASET_CLASSES={MAX_DATASET_CLASSES}")
        index = {c: i for i, c in enumerate(classes)}

        chosen: dict[str, dict[str, list[str]]] = {}
        if presplit:
            if not any(parts.get("val") for parts in files.values()):
                raise ValueError("val/ holds no images")
            no_val = sorted(c for c in classes if not files[c].get("val"))
            if no_val:
                raise ValueError(
                    f"class(es) {no_val} appear in train/ but not in val/; every class needs >= 1 val image "
                    "so its accuracy can be measured"
                )
            chosen = {c: {"train": files[c]["train"], "val": files[c].get("val", [])} for c in files}
        else:
            rng = random.Random(seed)
            for cls, parts in files.items():
                grouped: dict[str, list[str]] = {}
                for name in parts["all"]:
                    grouped.setdefault(parts_of(name)[-1], []).append(name)
                groups = list(grouped.values())
                if len(groups) < 2:
                    raise ValueError(
                        f"class {cls!r} has {len(groups)} image group(s) (images sharing a file name are "
                        "one group); each class needs >= 2 for a train/val split"
                    )
                rng.shuffle(groups)
                if subset_per_class:
                    kept, total = [], 0
                    for group in groups:
                        if total + len(group) > subset_per_class:
                            break
                        kept.append(group)
                        total += len(group)
                    if len(kept) < 2:
                        raise ValueError(f"class {cls!r}: subset_per_class={subset_per_class} < 2 groups")
                    groups = kept
                n_val = max(1, int(len(groups) * validation_split))
                chosen[cls] = {
                    "train": [name for group in groups[n_val:] for name in group],
                    "val": [name for group in groups[:n_val] for name in group],
                }

        out: dict[str, Any] = {"train_images": [], "train_targets": [], "val_images": [], "val_targets": [],
                               "train_ids": [], "val_ids": []}
        for cls, parts in chosen.items():
            for part in ("train", "val"):
                for member in parts[part]:
                    out[f"{part}_images"].append(_decode_member(archive, member))
                    out[f"{part}_targets"].append(index[cls])
                    out[f"{part}_ids"].append(member)
    out["classes"] = classes
    out["manifest"] = {
        "source": source,
        "sha256": hashlib.sha256(zip_bytes).hexdigest(),
        "bytes": len(zip_bytes),
        "layout": "train/ and val/ folders" if presplit else "class folders, seeded pair-grouped split",
        "validation_split": None if presplit else validation_split,
        "seed": None if presplit else seed,
        "subset_per_class": None if presplit else subset_per_class,
        "classes": classes,
        "per_class": {c: {"train": len(chosen[c]["train"]), "val": len(chosen[c]["val"])} for c in classes},
        "images_in_archive": len(names),
        "skipped": skipped,
    }
    return out


@dataclass
class ResNet50ClassificationPipeline:
    """``_runner`` maps a float tensor (N, 3, H, W) to logits (N, NUM_CLASSES); injectable for tests."""

    _runner: Callable[[Any], Any]
    _transform: Callable[[Image.Image], Any]
    device: str = "cpu"
    labels: tuple[str, ...] = ()
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ResNet50ClassificationPipeline:
        import timm
        import torch
        from timm.data import ImageNetInfo, create_transform, resolve_model_data_config

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        arch_name = MODEL_ID.split("/", 1)[1]
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")

        if (root / "model-config.json").is_file() and (root / WEIGHTS_FILE).is_file():
            from safetensors.torch import load_file
            with open(root / "model-config.json", encoding="utf-8") as fh:
                cfg = json.load(fh)
            num_classes = cfg.get("num_classes", len(cfg.get("class_names", [])))
            labels = tuple(cfg.get("class_names", [f"class_{i}" for i in range(num_classes)]))
            model = timm.create_model(arch_name, pretrained=False, num_classes=num_classes)
            model.load_state_dict(load_file(root / WEIGHTS_FILE, device=str(resolved_device)), strict=True)
            source = "fine-tuned-artifact"
            data_config = cfg.get("data_config") or resolve_model_data_config(model)
            transform = create_transform(**data_config, is_training=False)
        elif (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            with open(root / CONFIG_FILE, encoding="utf-8") as fh:
                config = json.load(fh)
            snapshot_name = f"{config['architecture']}.{config['pretrained_cfg']['tag']}"
            if snapshot_name != arch_name:
                raise ValueError(f"snapshot config names {snapshot_name!r}, expected {arch_name!r}")
            overlay = dict(config["pretrained_cfg"])
            overlay["file"] = str(root / WEIGHTS_FILE)  # 'file' takes precedence over hf_hub_id in timm
            model = timm.create_model(
                arch_name, pretrained=True, pretrained_cfg_overlay=overlay, num_classes=NUM_CLASSES
            )
            source = "local-snapshot"
            data_config = resolve_model_data_config(model)
            transform = create_transform(**data_config, is_training=False)
            info = ImageNetInfo(subset="imagenet-1k")
            labels = tuple(info.index_to_description(i) for i in range(info.num_classes()))
        elif allow_download:
            model = timm.create_model(
                _hub_reference(MODEL_ID, revision=MODEL_REVISION), pretrained=True, num_classes=NUM_CLASSES
            )
            source = "hf-hub"
            data_config = resolve_model_data_config(model)
            transform = create_transform(**data_config, is_training=False)
            info = ImageNetInfo(subset="imagenet-1k")
            labels = tuple(info.index_to_description(i) for i in range(info.num_classes()))
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        model = model.eval().to(resolved_device)

        def runner(batch: Any) -> Any:
            with torch.inference_mode():
                return model(batch.to(resolved_device))

        return cls(runner, transform, resolved_device, labels, source)

    @classmethod
    def fit(
        cls,
        train_images: Sequence[Image.Image],
        train_targets: Sequence[int],
        val_images: Sequence[Image.Image],
        val_targets: Sequence[int],
        class_names: Sequence[str],
        *,
        epochs: int = 1,
        batch_size: int = 4,
        learning_rate: float = 1e-4,
        weight_decay: float = 0.01,
        seed: int = 20260910,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        output_dir: str | Path | None = None,
        allow_download: bool = False,
        train_backbone: bool = True,
        provenance: Mapping[str, Any] | None = None,
    ) -> tuple[ResNet50ClassificationPipeline, dict[str, Any]]:
        """Fine-tune on custom classes in-process: a new linear classifier sized to ``class_names``.

        ``train_backbone=True`` (the default) is **full fine-tuning**: AdamW updates every parameter, the
        pretrained backbone included. ``train_backbone=False`` freezes everything except the new classifier
        (``model.get_classifier()``) and keeps every BatchNorm layer in eval mode, so its running statistics
        stay pretrained: head-only fine-tuning, where only ``fc.*`` changes. The training configuration, the
        trainable/frozen parameter counts and ``provenance`` (e.g. the dataset source, digest and split sizes)
        are returned under ``config`` and written into ``model-config.json`` with the artifact.
        """
        import timm
        import torch
        import torch.nn.functional as F
        from safetensors.torch import save_file
        from torch.utils.data import DataLoader, Dataset

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        num_classes = len(class_names)
        if num_classes < 2:
            raise ValueError(f"classification requires at least 2 classes, got {num_classes}")
        _check_training_config(epochs, batch_size, learning_rate, weight_decay)
        arch_name = MODEL_ID.split("/", 1)[1]

        # timm's training transforms (random-resized crop, flip, interpolation) use from Python's `random`, so
        # it is seeded too: with only torch seeded, two CPU runs with the same seed trained differently.
        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            with open(root / CONFIG_FILE, encoding="utf-8") as fh:
                config = json.load(fh)
            snapshot_name = f"{config['architecture']}.{config['pretrained_cfg']['tag']}"
            if snapshot_name != arch_name:
                raise ValueError(f"snapshot config names {snapshot_name!r}, expected {arch_name!r}")
            overlay = dict(config["pretrained_cfg"])
            overlay["file"] = str(root / WEIGHTS_FILE)
            model = timm.create_model(
                arch_name,
                pretrained=True,
                pretrained_cfg_overlay=overlay,
                num_classes=num_classes,
            )
        elif allow_download:
            model = timm.create_model(
                _hub_reference(MODEL_ID, revision=MODEL_REVISION),
                pretrained=True,
                num_classes=num_classes,
            )
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )

        model.to(resolved_device)
        data_config = timm.data.resolve_model_data_config(model)
        train_transform = timm.data.create_transform(**data_config, is_training=True)
        eval_transform = timm.data.create_transform(**data_config, is_training=False)

        class ImageDataset(Dataset):
            def __init__(self, imgs: Sequence[Image.Image], targets: Sequence[int], transform_fn: Any):
                self.imgs = list(imgs)
                self.targets = list(targets)
                self.transform_fn = transform_fn

            def __len__(self) -> int:
                return len(self.imgs)

            def __getitem__(self, idx: int) -> tuple[Any, int]:
                img = self.imgs[idx].convert("RGB")
                tensor = self.transform_fn(img)
                return tensor, self.targets[idx]

        train_loader = DataLoader(
            ImageDataset(train_images, train_targets, train_transform),
            batch_size=batch_size,
            shuffle=True,
        )
        val_loader = DataLoader(
            ImageDataset(val_images, val_targets, eval_transform),
            batch_size=batch_size,
            shuffle=False,
        )

        counts = set_trainable(model, train_backbone=train_backbone)
        config = {
            "method": counts["method"],
            "trainable": counts["trainable"],
            "trainable_parameters": counts["trainable_parameters"],
            "frozen_parameters": counts["frozen_parameters"],
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "seed": seed,
            "batchnorm_statistics": "updated in training" if train_backbone else "frozen (eval mode)",
            "optimizer": "AdamW",
            "loss": "cross-entropy",
            "train_samples": len(train_targets),
            "val_samples": len(val_targets),
            **({"dataset": dict(provenance)} if provenance else {}),
        }
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
        history = []

        for epoch in range(epochs):
            model.train()
            if not train_backbone:
                # RN-M3: a frozen backbone stays frozen. ResNet-50's BatchNorm running statistics update in
                # train() mode even with requires_grad False, so those layers are kept in eval() mode.
                for module in model.modules():
                    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                        module.eval()
            train_loss_sum, train_count = 0.0, 0
            for inputs, targets in train_loader:
                inputs = inputs.to(resolved_device)
                targets = targets.to(resolved_device)
                optimizer.zero_grad(set_to_none=True)
                outputs = model(inputs)
                loss = F.cross_entropy(outputs, targets)
                loss.backward()
                optimizer.step()
                train_loss_sum += float(loss.detach().cpu()) * targets.numel()
                train_count += targets.numel()

            model.eval()
            val_loss_sum, val_correct, val_count = 0.0, 0, 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs = inputs.to(resolved_device)
                    targets = targets.to(resolved_device)
                    outputs = model(inputs)
                    loss = F.cross_entropy(outputs, targets)
                    val_loss_sum += float(loss.detach().cpu()) * targets.numel()
                    val_correct += int((outputs.argmax(dim=-1) == targets).sum())
                    val_count += targets.numel()

            history.append({
                "epoch": epoch + 1,
                "train_loss": train_loss_sum / max(1, train_count),
                "val_loss": val_loss_sum / max(1, val_count),
                "val_accuracy": val_correct / max(1, val_count),
            })

        if output_dir is not None:
            out_path = Path(output_dir)
            out_path.mkdir(parents=True, exist_ok=True)
            safetensors_path = out_path / WEIGHTS_FILE
            tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
            save_file(tensors, safetensors_path)
            config_payload = {
                "architecture": "resnet50",
                "num_classes": num_classes,
                "class_names": list(class_names),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "data_config": data_config,
                "fine_tuning": config,
            }
            with open(out_path / "model-config.json", "w", encoding="utf-8") as fh:
                json.dump(config_payload, fh, indent=2)

        model.eval()

        def runner(batch: Any) -> Any:
            with torch.inference_mode():
                return model(batch.to(resolved_device))

        pipeline = cls(runner, eval_transform, resolved_device, tuple(class_names), "fine-tuned")
        return pipeline, {
            "history": history, "class_names": list(class_names), "device": resolved_device, "config": config
        }

    def _validate(self, images: Any, top_k: int) -> list[Image.Image]:
        max_classes = len(self.labels) if self.labels else NUM_CLASSES
        return _check_inputs(images, top_k, max_classes=max_classes)

    def predict(
        self, images: Image.Image | Sequence[Image.Image], top_k: int | None = None
    ) -> dict[str, Any]:
        """Classify images; ``score`` is a softmax score, not a calibrated probability."""
        import torch

        active_classes = len(self.labels) if self.labels else NUM_CLASSES
        resolved_top_k = min(DEFAULT_TOP_K, active_classes) if top_k is None else top_k
        batch_images = self._validate(images, resolved_top_k)
        batch = torch.stack([self._transform(image.convert("RGB")) for image in batch_images])
        logits = self._runner(batch)
        expected_classes = active_classes
        if not isinstance(logits, torch.Tensor) or logits.shape != (len(batch_images), expected_classes):
            raise RuntimeError(f"runner must return a tensor of shape (batch, {expected_classes})")
        scores = torch.softmax(logits.float(), dim=-1).cpu()
        values, indices = torch.topk(scores, k=resolved_top_k, dim=-1)
        predictions = []
        for image_values, image_indices in zip(values.tolist(), indices.tolist(), strict=True):
            image_values = [float(s) for s in image_values]
            ranked = [
                {"label": self.labels[i] if i < len(self.labels) else str(i), "index": i, "score": s}
                for s, i in zip(image_values, image_indices, strict=True)
            ]
            best = ranked[0]
            predictions.append(
                {"predicted_index": best["index"], "predicted_label": best["label"], "top_k": ranked}
            )
        return {
            "predictions": predictions,
            "top_k": resolved_top_k,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `767268603ca0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ResNet50ClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: pin, stage and verify the model (you may run it without reading it)
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "resnet50-a1",
  "modelId": "timm/resnet50.a1_in1k",
  "revision": "767268603ca0cb0bfe326fa87277f19c419566ef",
  "files": [
    {
      "path": "README.md",
      "bytes": 38435,
      "sha256": "1bf90a24888cff1da80e2e7b2e341fad1e4e448c19d92eb5bc08468393251721"
    },
    {
      "path": "config.json",
      "bytes": 756,
      "sha256": "ff00d936f02c3b5f9f80a169d651ed28f9b6dc9cb1fd0464f92da4761caaac50"
    },
    {
      "path": "model.safetensors",
      "bytes": 102469840,
      "sha256": "773525d5821de224f8f30c33377b7a795d7863e08522698200d3217d3f2a41bb"
    }
  ],
  "totalBytes": 102509031
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ResNet50ClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'timm/resnet50.a1_in1k', 'revision': '767268603ca0cb0bfe326fa87277f19c419566ef', 'license': 'apache-2.0', 'files': 3, 'total_bytes': 102509031}


{'weights_dir': '/content/weights/resnet50-a1', 'fetched': ['README.md', 'config.json', 'model.safetensors']}


{'verified_files': 3, 'revision': '767268603ca0cb0bfe326fa87277f19c419566ef'}


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a deterministic 256×256 RGB gradient built in code (red ramps left to right, green top to bottom, blue is their mean), so it needs no download and its SHA-256 is printed for the record. A gradient is not a photograph of any ImageNet class, so it has **no ground truth**: whatever label the model returns is a sanity check that the input contract, preprocessing and forward pass work, not a correctness measurement. BYOD is optional and disabled by default; when enabled, upload one image file (or set `BYOD_IMAGE_PATH` outside Colab) and, if you know its ImageNet-1k class index (0–999), set `GROUND_TRUTH_INDEX` so the evaluation step can compute `top_k_accuracy`. Leave it at `-1` when the label is unknown. A file that is not a decodable image is refused with its name. Look for a dictionary naming the sample kind, its size and digest, and whether a ground-truth index was supplied.

In [ ]:
import hashlib
import io
import os

import numpy as np
from PIL import Image, UnidentifiedImageError

USE_BYOD = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ''  # @param {type:"string"}
GROUND_TRUTH_INDEX = -1  # @param {type:"integer"}
SAMPLE_SIDE = 256

if USE_BYOD:
    if BYOD_IMAGE_PATH:
        image_name = os.path.basename(BYOD_IMAGE_PATH)
        with open(BYOD_IMAGE_PATH, 'rb') as handle:
            image_bytes = handle.read()
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one image file (got {len(uploaded)}). Outside Colab, set BYOD_IMAGE_PATH to a file path instead.')
        image_name, image_bytes = next(iter(uploaded.items()))
    try:
        image = Image.open(io.BytesIO(image_bytes))
        image.load()
    except (UnidentifiedImageError, Image.DecompressionBombError, OSError) as exc:
        raise ValueError(f'{image_name}: not a decodable image ({type(exc).__name__}). Use a PNG, JPEG, WebP or BMP file and run this cell again.') from None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic gradient: no randomness, so no seed is needed and the digest is stable.
    ramp = np.linspace(0.0, 255.0, SAMPLE_SIDE)
    red = np.tile(ramp, (SAMPLE_SIDE, 1))
    green = red.T
    blue = (red + green) / 2.0
    array = np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8)
    image = Image.fromarray(array, mode='RGB')
    image_name = f'synthetic_gradient_{SAMPLE_SIDE}.png'
    sample_kind = 'synthetic'

if GROUND_TRUTH_INDEX != -1 and not 0 <= GROUND_TRUTH_INDEX < NUM_CLASSES:
    raise ValueError(f'GROUND_TRUTH_INDEX must be -1 (unknown) or an ImageNet-1k class index in 0..{NUM_CLASSES - 1}, got {GROUND_TRUTH_INDEX}.')
ground_truth = None if GROUND_TRUTH_INDEX == -1 else GROUND_TRUTH_INDEX
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': sample_sha256, 'ground_truth_index': ground_truth})

{'sample_kind': 'synthetic', 'name': 'synthetic_gradient_256.png', 'mode': 'RGB', 'size': (256, 256), 'rgb_sha256': 'e38db00bd95756e739b8d2f6a69de029b011193c1e6a2a589aa546c4631adbdf', 'ground_truth_index': None}


<isolated cell 4>:37: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(array, mode='RGB')


## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `predict` applies — type, batch size 1..`MAX_BATCH`, image side 1..`MAX_IMAGE_SIDE` px, `top_k` 1..`NUM_CLASSES` — and returns an **input manifest** naming the schema and ceilings, each input's observed mode and size, and the verdict. The manifest is written to `outputs/resnet50_classification_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately oversized image and records the pipeline's own error message as a finding. Inside the pipeline every accepted image is converted to RGB, resized to 235 px and center-cropped to 224×224 (`crop_pct = 0.95`, bicubic), so content near the border is cropped away; nothing else is dropped or altered.

**Predict before running:** the oversized probe is 4097 px wide. Will the manifest's verdict for *your* image change because of it?

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'NUM_CLASSES': NUM_CLASSES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH}})
input_manifest = validate_inputs(image, top_k=5, names=[image_name])
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8)))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/resnet50_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

{'ceilings': {'NUM_CLASSES': 1000, 'MAX_IMAGE_SIDE': 4096, 'MAX_BATCH': 64}}
{
  "schema": {
    "input": "PIL.Image.Image or a sequence of them; any mode, converted to RGB",
    "image_side_px": [
      1,
      4096
    ],
    "batch": [
      1,
      64
    ],
    "top_k": [
      1,
      1000
    ],
    "preprocessing": "resize to 235 px, center-crop 224x224 (crop_pct 0.95, bicubic), ImageNet normalisation"
  },
  "inputs": [
    {
      "id": "synthetic_gradient_256.png",
      "mode": "RGB",
      "size": [
        256,
        256
      ]
    }
  ],
  "top_k": 5,
  "verdict": "accepted",
  "findings": [
    {
      "input": "oversized-probe",
      "verdict": "rejected",
      "message": "image side outside 1..MAX_IMAGE_SIDE=4096 px: (4097, 8)"
    }
  ],
  "model_id": "timm/resnet50.a1_in1k",
  "model_revision": "767268603ca0cb0bfe326fa87277f19c419566ef"
}


**What to notice:** `verdict: accepted` for the sample, and one finding that names the ceiling the probe broke (`MAX_IMAGE_SIDE=4096`).

<details><summary>Check your reasoning</summary>No. The probe is validated separately and its rejection is recorded as a finding; the sample's own verdict stays `accepted`. A rejected input never reaches the model: `predict` raises the same error.</details>

## 6. Classify

`predict` returns, per image, `predicted_index`/`predicted_label` and a `top_k` list of `{label, index, score}` entries **ordered by descending score** — rank position is the class ordering, and the exported files preserve it. The decision rule is `argmax` over the 1000 softmax scores (`decision_rule` in the result); the pipeline ships no acceptance threshold, and `score` is a softmax over uncalibrated logits, **not a calibrated probability**. A deployment that needs an abstain option must choose its own score cut-off on its own labelled data — downstream calibration is the caller's responsibility. Inference is deterministic given the same weights, device and library versions (no sampling, `model.eval()`); CPU, GPU and cuDNN kernel choices can reorder near-tied classes.

**Predict before running:** the gradient shows no object. Will the top-1 score be high (above 0.5) or low? Will the five classes be related to each other?

In [ ]:
result = pipe.predict(image, top_k=5)
prediction = result['predictions'][0]
print({'decision_rule': result['decision_rule'], 'predicted_index': prediction['predicted_index'], 'predicted_label': prediction['predicted_label'], 'device': result['device'], 'source': result['source']})
for rank, item in enumerate(prediction['top_k'], start=1):
    print(f"{rank:>2}. index {item['index']:>4}  score {item['score']:.4f}  {item['label']}")

{'decision_rule': 'argmax', 'predicted_index': 916, 'predicted_label': 'web site, website, internet site, site', 'device': 'cuda:0', 'source': 'local-snapshot'}
 1. index  916  score 0.0392  web site, website, internet site, site
 2. index  854  score 0.0382  theater curtain, theatre curtain
 3. index  644  score 0.0321  matchstick
 4. index  149  score 0.0221  dugong, Dugong dugon
 5. index  499  score 0.0209  cleaver, meat cleaver, chopper


**What to notice:** a low top-1 score spread across unrelated classes.

<details><summary>Check your reasoning</summary>Low and unrelated. In a local Windows CPU check of this notebook version (2026-10-09) the top-1 was `web site, website, internet site, site` with score 0.0392, followed by `theater curtain` (0.0382), `matchstick`, `dugong` and `cleaver`, all between 0.02 and 0.04. A softmax always sums to 1 over the 1000 classes, so the model must name *something* even when nothing matches; a low, flat top-5 is how that looks. It is not a probability that the image shows a web site. A GPU run can reorder these near-tied classes.</details>

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. When a ground-truth class index was supplied in Section 4 it carries `top_k_accuracy` (the repository's metric helper) at k=1 and k=5 with the verdict `sample-sanity` — a single-image tutorial metric with no dispersion estimate. On the synthetic default sample no metric exists, so the verdict is `not-measurable` and the report states what would make the task measurable: labelled photographs with ImageNet-1k class indices, for example a held-out sample of your own data scored against its majority-class baseline, or the ImageNet-1k validation set (whose upstream 80.38 % top-1 / 94.60 % top-5 at 224 px is quoted from the upstream card, not measured here). The report is written to `outputs/resnet50_classification_evaluation_report.json`.

**Predict before running:** which verdict will the report give for the gradient, and why?

In [ ]:
targets = None if ground_truth is None else [ground_truth]
report = evaluation_report(result, targets, sample_kind=sample_kind)
with open('outputs/resnet50_classification_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No ground-truth class index was supplied, so top_k_accuracy is not computed; the prediction above is sanity evidence only.')

{
  "task": "imagenet-1k single-label classification",
  "decision_rule": "argmax",
  "sample_kind": "synthetic",
  "n_predictions": 1,
  "baselines": [],
  "model_id": "timm/resnet50.a1_in1k",
  "model_revision": "767268603ca0cb0bfe326fa87277f19c419566ef",
  "metrics": [],
  "verdict": "not-measurable",
  "reason": "no ground-truth class index was supplied for the evaluated images",
  "needs": "labelled photographs with ImageNet-1k class indices (0-999), e.g. a held-out sample of your own data, scored with top_k_accuracy against the majority-class baseline of that sample"
}
No ground-truth class index was supplied, so top_k_accuracy is not computed; the prediction above is sanity evidence only.


**What to notice:** `verdict: not-measurable`, an empty `metrics` list, and a `needs` field saying what labelled data would be required.

<details><summary>Check your reasoning</summary>`not-measurable`: accuracy compares a prediction with a true label, and a gradient has none. Supplying a photograph and its index through BYOD switches the verdict to `sample-sanity` — still one image, so still not a measurement of the model.</details>

## 8. Fine-tuning data: acquire, validate and split

From here the notebook adapts the model to **new classes**. That needs labelled images, split into a **training** set the model learns from and a **held-out** set it never sees during training, so the evaluation in Section 10 measures something the model could not have memorised.

- **Default (`USE_BYOD_DATASET = False`):** downloads [`Cleanlab/cifar-10-subset`](https://huggingface.co/datasets/Cleanlab/cifar-10-subset) (MIT licence, 986,707 bytes) and refuses it unless its SHA-256 matches. It holds 400 CIFAR-10 images of **32×32 px** in two classes, `frog` and `truck`; each photo is stored twice, as an original and a darkened copy. The split is **pair-grouped**: images of one class with the same file name (`original_images/frog/image_7.png` and `darkened_images/frog/image_7.png`) form one group, and a group always lands on one side, so a held-out photo's darkened twin is never in training (otherwise the model could score it by memory). With `SPLIT_SEED`, `SUBSET_PER_CLASS = 100` images (50 groups) per class are drawn and 20 % of the groups (10 per class) are held out: **160 training and 40 held-out images**. The model's preprocessing upscales every image to 224 px. The cell prints `held_out_sharing_a_file_name_with_train`, which the grouped split keeps at 0. If the download fails, the cell says so and falls back to a synthetic two-class stripes dataset (8 training, 4 held-out images); the results below then describe stripes.
- **Bring Your Own Data (`USE_BYOD_DATASET = True`):** upload a `.zip` (or set `BYOD_DATASET_PATH`) of `<class>/<image>` folders, split here per class with `SPLIT_SEED`, or of `train/<class>/` and `val/<class>/` folders used as given. `load_image_zip` decodes every image, checks it against `MAX_IMAGE_SIDE`, and refuses — before any training, naming the file or class and the rule — an image outside a class folder, a validation class missing from `train/`, a class with fewer than 2 image groups, fewer than 2 classes, an undecodable or oversized image, or an archive over its limits (see the Prerequisites). Other files are skipped and listed.

**Predict before running:** with 20 held-out images per class, by how many percentage points does one wrong prediction move the held-out accuracy? And how many independent photos are those 40 images?

In [ ]:
import hashlib
import os
import urllib.request

USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}
SAMPLE_DATASET_URL = 'https://huggingface.co/datasets/Cleanlab/cifar-10-subset/resolve/bb5a7aabf1d14d2d1e3e49d0d8f917bda3622f75/CIFAR-10-subset.zip'
SAMPLE_DATASET_SHA256 = '66f90a4f87d865e8eb653b62f10e754684075a32314177de76832349d4b1fb19'
VALIDATION_SPLIT = 0.2
SUBSET_PER_CLASS = 100  # images per class kept from the tutorial dataset (50 photo pairs: 80 train + 20 held out)

if USE_BYOD_DATASET:
    if BYOD_DATASET_PATH:
        zip_name = os.path.basename(BYOD_DATASET_PATH)
        with open(BYOD_DATASET_PATH, 'rb') as handle:
            zip_bytes = handle.read()
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one dataset .zip archive (got {len(uploaded)}). Outside Colab, set BYOD_DATASET_PATH instead.')
        zip_name, zip_bytes = next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'):
        raise ValueError(f'BYOD archive must be a .zip file, got {zip_name}')
    dataset = load_image_zip(zip_bytes, seed=SPLIT_SEED, validation_split=VALIDATION_SPLIT, source=f'user upload: {zip_name}')
    dataset_kind = 'BYOD'
else:
    zip_bytes, download_error = None, None
    try:
        request = urllib.request.Request(SAMPLE_DATASET_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(request, timeout=30) as response:
            zip_bytes = response.read()
    except OSError as exc:  # no network, DNS failure, HTTP error, timeout
        download_error = exc
    if zip_bytes is not None:
        actual_sha = hashlib.sha256(zip_bytes).hexdigest()
        if actual_sha != SAMPLE_DATASET_SHA256:
            raise ValueError(f'The tutorial dataset from {SAMPLE_DATASET_URL} has SHA-256 {actual_sha}, expected {SAMPLE_DATASET_SHA256}: refusing to train on it. Run this cell again; if it repeats, the download is being altered.')
        dataset = load_image_zip(zip_bytes, seed=SPLIT_SEED, validation_split=VALIDATION_SPLIT, subset_per_class=SUBSET_PER_CLASS, source='Cleanlab/cifar-10-subset @ bb5a7aab (MIT)')
        dataset_kind = 'tutorial dataset'
    else:
        print(f'Warning: the tutorial dataset could not be downloaded from {SAMPLE_DATASET_URL} ({download_error}); falling back to the deterministic synthetic stripes dataset (2 classes, 8 training and 4 held-out images). Results below then describe stripes, not photographs.')
        dataset = synthetic_stripes_dataset()
        dataset_kind = 'synthetic fallback'

CUSTOM_CLASSES = dataset['classes']
train_images, train_targets = dataset['train_images'], dataset['train_targets']
val_images, val_targets = dataset['val_images'], dataset['val_targets']
dataset_manifest = dataset['manifest']
# Two held-out images that share a file name with a training image of the same class may be copies of one picture
# (the tutorial archive stores each photo twice: original_images/ and darkened_images/).
train_keys = {(t, os.path.basename(i)) for t, i in zip(train_targets, dataset['train_ids'])}
shared_names = [i for t, i in zip(val_targets, dataset['val_ids']) if (t, os.path.basename(i)) in train_keys]
dataset_provenance = {
    'kind': dataset_kind,
    'source': dataset_manifest['source'],
    'sha256': dataset_manifest['sha256'],
    'classes': CUSTOM_CLASSES,
    'per_class': dataset_manifest['per_class'],
    'split_seed': dataset_manifest.get('seed'),
    'subset_per_class': dataset_manifest.get('subset_per_class'),
    'validation_split': dataset_manifest.get('validation_split'),
    'held_out_sharing_a_file_name_with_train': len(shared_names),
}
print({'dataset': dataset_kind, 'source': dataset_manifest['source'], 'sha256': dataset_manifest['sha256'], 'layout': dataset_manifest['layout'], 'image_size_px': list(train_images[0].size)})
print(f"{'class':<32}{'train':>6}{'held out':>10}")
for name, counts in dataset_manifest['per_class'].items():
    print(f"{name:<32}{counts['train']:>6}{counts['val']:>10}")
print({'train_images': len(train_images), 'held_out_images': len(val_images), 'held_out_sharing_a_file_name_with_train': len(shared_names), 'skipped_files': dataset_manifest['skipped'][:10]})

{'dataset': 'tutorial dataset', 'source': 'Cleanlab/cifar-10-subset @ bb5a7aab (MIT)', 'sha256': '66f90a4f87d865e8eb653b62f10e754684075a32314177de76832349d4b1fb19', 'layout': 'class folders, seeded pair-grouped split', 'image_size_px': [32, 32]}
class                            train  held out
frog                                80        20
truck                               80        20
{'train_images': 160, 'held_out_images': 40, 'held_out_sharing_a_file_name_with_train': 0, 'skipped_files': []}


**What to notice:** the dataset digest, `image_size_px: [32, 32]`, 80 training and 20 held-out images per class, and `held_out_sharing_a_file_name_with_train: 0`.

<details><summary>Check your reasoning</summary>By 2.5 points: 40 held-out images, so each one is 1/40 of the accuracy. But they are only 20 photos, each held out with its darkened copy, and the two copies are usually classified alike, so the 40 images carry roughly the evidence of 20 independent photos. Keep that in mind when Section 10 prints an interval computed as if all 40 were independent.</details>

## 9. Fine-tune

`pipe.fit(...)` builds the network from the verified snapshot, replaces the 1000-class head with a new linear classifier sized to the target classes (`len(class_names)`, here 2), and trains with `torch.optim.AdamW` (Adam with decoupled weight decay) on the **cross-entropy** loss between the softmax and the true class. Each run starts again from the pretrained snapshot. The form fields are the whole training configuration:

- `TRAINABLE`: `'head'` (default) is **head-only fine-tuning** — the pretrained backbone is frozen and only the new classifier is trained (4,098 parameters for 2 classes) and the backbone's BatchNorm layers stay in evaluation mode, so their running statistics stay pretrained; `'all'` is **full fine-tuning** — every parameter, the backbone included, is updated and BatchNorm statistics are re-estimated from each batch of `BATCH_SIZE` images (about 23.5 M parameters; Section 12).
- `EPOCHS` (passes over the training images), `BATCH_SIZE`, `LEARNING_RATE`, `WEIGHT_DECAY` and `TRAIN_SEED`, which seeds PyTorch (head initialisation, shuffling) and Python's `random` (the training augmentation), so a CPU run with the same settings repeats exactly; GPU kernels can still differ slightly.

The cell prints the trainable and frozen parameter counts **before** training and checks that `fit` trained exactly that set. The configuration, the counts and the dataset provenance (source, SHA-256, classes, split) are written into `model-config.json` with the artifact and into the result file. No external worker, CLI subprocess or unpinned dependency is involved. The per-epoch history reports training loss, validation loss and validation accuracy on the held-out images.

**Predict before running:** after one epoch on 160 images, will the validation accuracy be above the 50 % that always answering one class would get? How many parameters will the default run train?

In [ ]:
TRAINABLE = 'head'  # @param ["all", "head"]
EPOCHS = 1  # @param {type:"integer"}
BATCH_SIZE = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
WEIGHT_DECAY = 0.01  # @param {type:"number"}
TRAIN_SEED = 20260910  # @param {type:"integer"}

if TRAINABLE not in ('all', 'head'):
    raise ValueError(f"TRAINABLE must be 'all' (full fine-tuning) or 'head' (head only), got {TRAINABLE!r}")
train_backbone = TRAINABLE == 'all'
planned = trainable_parameter_counts(len(CUSTOM_CLASSES), train_backbone=train_backbone)
print({'method': planned['method'], 'trainable_parameters': planned['trainable_parameters'], 'frozen_parameters': planned['frozen_parameters'], 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'seed': TRAIN_SEED})

ft_output_dir = 'outputs/resnet50_classification_finetuned'
fine_tuned_pipe, train_meta = pipe.fit(
    train_images=train_images,
    train_targets=train_targets,
    val_images=val_images,
    val_targets=val_targets,
    class_names=CUSTOM_CLASSES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    seed=TRAIN_SEED,
    train_backbone=train_backbone,
    weights_dir=WEIGHTS_DIR,
    output_dir=ft_output_dir,
    provenance=dataset_provenance,
)
finetune_config = train_meta['config']
if (finetune_config['trainable_parameters'], finetune_config['frozen_parameters']) != (planned['trainable_parameters'], planned['frozen_parameters']):
    raise RuntimeError(f'fit trained a different parameter set than the one printed above: {finetune_config}')
print({'fine_tuning': 'complete', 'method': finetune_config['method'], 'device': train_meta['device'], 'classes': CUSTOM_CLASSES, 'train_samples': finetune_config['train_samples'], 'val_samples': finetune_config['val_samples']})
for row in train_meta['history']:
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in row.items()})

{'method': 'head-only fine-tuning', 'trainable_parameters': 4098, 'frozen_parameters': 23508032, 'epochs': 1, 'batch_size': 4, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'seed': 20260910}


{'fine_tuning': 'complete', 'method': 'head-only fine-tuning', 'device': 'cuda:0', 'classes': ['frog', 'truck'], 'train_samples': 160, 'val_samples': 40}
{'epoch': 1, 'train_loss': 0.6823, 'val_loss': 0.6285, 'val_accuracy': 0.85}


**What to notice:** `head-only fine-tuning` with `trainable_parameters` 4,098 (2,048 features × 2 classes + 2 biases) and `frozen_parameters` 23,508,032 (the whole pretrained backbone), then one history row with the training loss, the validation loss and the validation accuracy.

<details><summary>Check your reasoning</summary>Yes, in the local check — but the losses show the head has only just started to learn. In a local Windows CPU check of this notebook version (torch 2.14.0, defaults, 2026-10-09) one epoch gave training loss 0.68, validation loss 0.63 (an untrained two-class head scores about 0.69) and validation accuracy 34/40. The frozen backbone already describes images well (it was trained on ImageNet), so a linear head on its features only has to learn which direction separates frogs from trucks — 4,098 numbers. One epoch at learning rate 1e-4 moves a fresh head only a little, so the softmax scores stay close to 0.5 even when the ranking is right. With `SPLIT_SEED` 0, 1 and 2 the same check scored 34/40, 37/40 and 32/40. A GPU run can differ by an image or two.</details>

## 10. Fresh-boundary reload, equivalence check and held-out evaluation

Loading a file is not proof that it holds the model you evaluated. `ResNet50ClassificationPipeline.from_pretrained` loads `model.safetensors` and `model-config.json` from `outputs/resnet50_classification_finetuned` as a fresh deployment would (architecture rebuilt, weights restored with `strict=True`, class names from the config); the cell then predicts every held-out image with **both** the reloaded and the in-memory model and requires the same labels and a softmax-score difference within `EQUIVALENCE_TOLERANCE` (`equivalent: True`). It stops with an explanation otherwise. The artifact's SHA-256 digests are recorded.

The held-out evaluation is a **tutorial metric on a handful of images, not a benchmark**. `finetune_evaluation_report` reports the counts, the accuracy with its **95 % Wilson interval**, the **majority-class baseline** (the accuracy of always answering the most common held-out class) and per-class counts and misclassified images. Its verdict is always `sample-sanity`; `comparison_to_baseline` is `above-baseline` only when the whole interval lies above the baseline, `below-baseline` when it lies below, and otherwise `indistinguishable-from-baseline`. Files: `outputs/resnet50_classification_validation_predictions.csv` and `outputs/resnet50_classification_finetuned_evaluation_report.json`.

Because the 40 held-out images are 20 photos in two copies, the interval treats correlated images as independent and is somewhat **too narrow**; read it as a lower bound on the uncertainty.

**Predict before running:** if the model gets 26 of the 40 held-out images right (65 %), does the 95 % interval exclude the 50 % baseline? What about 30 of 40?

In [ ]:
import csv
import hashlib

EQUIVALENCE_TOLERANCE = 1e-4  # largest allowed softmax-score difference between the reloaded and in-memory model

reloaded_pipe = ResNet50ClassificationPipeline.from_pretrained(weights_dir=ft_output_dir)

# Fresh-boundary check: the reloaded artifact must reproduce the in-memory model on every held-out image.
predicted_indices, predicted_scores, label_agreement, max_score_difference = [], [], 0, 0.0
for img in val_images:
    reloaded = reloaded_pipe.predict(img)['predictions'][0]
    in_memory = fine_tuned_pipe.predict(img)['predictions'][0]
    label_agreement += reloaded['predicted_index'] == in_memory['predicted_index']
    in_memory_scores = {item['index']: item['score'] for item in in_memory['top_k']}
    for item in reloaded['top_k']:
        max_score_difference = max(max_score_difference, abs(item['score'] - in_memory_scores.get(item['index'], 0.0)))
    predicted_indices.append(reloaded['predicted_index'])
    predicted_scores.append(reloaded['top_k'][0]['score'])
equivalence = {'images': len(val_images), 'label_agreement': label_agreement, 'max_score_difference': max_score_difference, 'tolerance': EQUIVALENCE_TOLERANCE}
equivalence['equivalent'] = label_agreement == len(val_images) and max_score_difference <= EQUIVALENCE_TOLERANCE
print({'fresh_boundary_source': reloaded_pipe.source, **equivalence})
if not equivalence['equivalent']:
    raise RuntimeError(f'The reloaded artifact does not reproduce the in-memory model ({equivalence}). Re-run Section 9; if it repeats, the export is broken and must not be used.')
artifact_sha256 = {}
for name in ('model.safetensors', 'model-config.json'):
    with open(os.path.join(ft_output_dir, name), 'rb') as handle:
        artifact_sha256[name] = hashlib.sha256(handle.read()).hexdigest()

# Held-out evaluation of the reloaded artifact, with the uncertainty that n images allow.
finetuned_eval_report = finetune_evaluation_report(val_targets, predicted_indices, CUSTOM_CLASSES, item_ids=dataset['val_ids'], scores=predicted_scores, sample_kind=dataset_kind)
finetuned_eval_report.update({'fresh_boundary_source': reloaded_pipe.source, 'equivalence': equivalence, 'artifact_sha256': artifact_sha256, 'fine_tuning_method': finetune_config['method']})
with open('outputs/resnet50_classification_finetuned_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(finetuned_eval_report, handle, indent=2, ensure_ascii=False)
with open('outputs/resnet50_classification_validation_predictions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['sample_index', 'item_id', 'true_label', 'predicted_label', 'correct', 'confidence'])
    for k, (item_id, target, predicted, score) in enumerate(zip(dataset['val_ids'], val_targets, predicted_indices, predicted_scores)):
        writer.writerow([k, item_id, CUSTOM_CLASSES[target], CUSTOM_CLASSES[predicted], target == predicted, f'{score:.6f}'])

metrics = finetuned_eval_report['metrics']
low, high = metrics['accuracy_wilson_95']
print(f"Held-out images: {finetuned_eval_report['n']} ({dataset_kind}) - a tutorial metric, not a benchmark")
print(f"Accuracy:        {finetuned_eval_report['correct']}/{finetuned_eval_report['n']} = {metrics['accuracy']:.1%}   95% Wilson interval {low:.1%} to {high:.1%}")
print(f"Majority-class baseline: {metrics['majority_baseline_accuracy']:.1%} (always answering '{metrics['majority_baseline_class']}')")
print(f"Verdict: {finetuned_eval_report['verdict']}; comparison to baseline: {finetuned_eval_report['comparison_to_baseline']}")
for name, counts in finetuned_eval_report['per_class'].items():
    print(f"  {name:<32}{counts['correct']}/{counts['total']} correct")
print({'misclassified': finetuned_eval_report['misclassified']})

run_history = globals().get('run_history', [])
run_history.append({'method': finetune_config['method'], 'trainable_parameters': finetune_config['trainable_parameters'], 'epochs': finetune_config['epochs'], 'learning_rate': finetune_config['learning_rate'], 'split_seed': dataset_provenance['split_seed'], 'dataset': dataset_kind, 'final_val_loss': round(train_meta['history'][-1]['val_loss'], 4), 'correct': finetuned_eval_report['correct'], 'n': finetuned_eval_report['n'], 'wilson_95': [round(low, 3), round(high, 3)], 'comparison_to_baseline': finetuned_eval_report['comparison_to_baseline']})

{'fresh_boundary_source': 'fine-tuned-artifact', 'images': 40, 'label_agreement': 40, 'max_score_difference': 0.0, 'tolerance': 0.0001, 'equivalent': True}


Held-out images: 40 (tutorial dataset) - a tutorial metric, not a benchmark
Accuracy:        34/40 = 85.0%   95% Wilson interval 70.9% to 92.9%
Majority-class baseline: 50.0% (always answering 'frog')
Verdict: sample-sanity; comparison to baseline: above-baseline
  frog                            14/20 correct
  truck                           20/20 correct
{'misclassified': [{'id': 'CIFAR-10-subset/original_images/frog/image_42.png', 'true_label': 'frog', 'predicted_label': 'truck', 'score': 0.513999342918396}, {'id': 'CIFAR-10-subset/darkened_images/frog/image_42.png', 'true_label': 'frog', 'predicted_label': 'truck', 'score': 0.5224961042404175}, {'id': 'CIFAR-10-subset/original_images/frog/image_9.png', 'true_label': 'frog', 'predicted_label': 'truck', 'score': 0.5021129846572876}, {'id': 'CIFAR-10-subset/darkened_images/frog/image_9.png', 'true_label': 'frog', 'predicted_label': 'truck', 'score': 0.501523494720459}, {'id': 'CIFAR-10-subset/original_images/frog/image_65.png', 'true

**What to notice:** `equivalent: True` with a score difference near 0; then `n`, the interval, the baseline and the comparison word, not just the accuracy, and which images were misclassified.

<details><summary>Check your reasoning</summary>26/40 = 65 % has a 95 % Wilson interval of about 49.5 % to 78 %: its lower end sits just under the baseline, so `indistinguishable-from-baseline`. 30/40 = 75 % gives about 60 % to 86 %: `above-baseline`. In the local CPU check of this version the default run scored 34/40 (interval about 71 % to 93 %, `above-baseline`; all 20 trucks right, 6 frogs called trucks — 3 photos, each in both copies, with scores just above 0.5) and the reload matched exactly (difference 0.0); with `SPLIT_SEED` 0, 1 and 2 it scored 34/40, 37/40 and 32/40, each `above-baseline`. That is consistent evidence that the adapted head separates these two classes on this data — but on 20 photos of one archive, not a claim about frogs and trucks in general.</details>

## 11. Export outputs and provenance

*Engineering and reproducibility.* Machine-readable JSON preserves the full prediction (argmax decision and the rank-ordered top-k scores), the evaluation report, the fine-tuning configuration (method, trainable and frozen parameter counts, epochs, batch size, learning rate, weight decay, seed), the dataset manifest (source, SHA-256, classes, split sizes, skipped files), the training history, the held-out evaluation with its interval and the equivalence check, the artifact digests, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `timm`, device). The rank-ordered top-k table and the held-out predictions are written as CSV files so class ordering survives downstream use. The fine-tuned artifact is in `outputs/resnet50_classification_finetuned/`. No credentials are recorded.

In [ ]:
import csv

payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'fine_tuning': {
        'config': finetune_config,
        'dataset': dataset_manifest,
        'history': train_meta['history'],
        'classes': CUSTOM_CLASSES,
        'evaluation': finetuned_eval_report,
        'artifact_dir': ft_output_dir,
        'artifact_sha256': artifact_sha256,
    },
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': sample_sha256, 'ground_truth_index': ground_truth},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'timm': timm.__version__,
        'device': pipe.device,
    },
}
with open('outputs/resnet50_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/resnet50_classification_top_k.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'index', 'label', 'score'])
    for rank, item in enumerate(prediction['top_k'], start=1):
        writer.writerow([image_name, rank, item['index'], item['label'], f"{item['score']:.6f}"])
print({'outputs': sorted(os.listdir('outputs')), 'finetuned': sorted(os.listdir(ft_output_dir))})
print(['outputs/resnet50_classification_finetuned/model.safetensors', 'outputs/resnet50_classification_finetuned/model-config.json', 'outputs/resnet50_classification_validation_predictions.csv', 'outputs/resnet50_classification_finetuned_evaluation_report.json'])

{'outputs': ['resnet50_classification_evaluation_report.json', 'resnet50_classification_finetuned', 'resnet50_classification_finetuned_evaluation_report.json', 'resnet50_classification_input_manifest.json', 'resnet50_classification_result.json', 'resnet50_classification_top_k.csv', 'resnet50_classification_validation_predictions.csv'], 'finetuned': ['model-config.json', 'model.safetensors']}
['outputs/resnet50_classification_finetuned/model.safetensors', 'outputs/resnet50_classification_finetuned/model-config.json', 'outputs/resnet50_classification_validation_predictions.csv', 'outputs/resnet50_classification_finetuned_evaluation_report.json']


**What to notice:** six files in `outputs/` plus the two-file artifact directory, and in `outputs/resnet50_classification_result.json` a `fine_tuning.config` block you could rerun from.

## 12. Your turn — change one thing: train every layer

**Predict → Change one thing → Run → Observe → Explain.**

1. **Predict:** with every layer trainable (`TRAINABLE = 'all'`, 23,512,130 trainable parameters instead of 4,098), will one epoch at the same learning rate do better or worse than the head-only run on the 40 held-out images? Write your guess down.
2. **Change one thing:** in Section 9 set `TRAINABLE = 'all'` and nothing else.
3. **Run:** select the Section 9 cell and choose **Runtime → Run after** (it re-runs Sections 9–12; the artifact and result files are overwritten with the full fine-tuning run, which `fine_tuning.config.method` records). It is slower: 31 s instead of 5 s for the training in the local CPU check.
4. **Observe:** this cell prints one row per run in this session — method, trainable parameters, final validation loss, the held-out count with its interval and the comparison to the baseline.
5. **Explain:** why can training more parameters give a *worse* model here, and does the held-out set have enough images to tell the two methods apart?

<details><summary>Check your reasoning</summary>Worse, in the local CPU check of this notebook version: one epoch of full fine-tuning ended with validation loss 0.68 (head-only: 0.63) and the held-out count fell from 34/40 to **21/40 — no better than always answering one class** (frog 5/20, truck 16/20; interval about 38 % to 67 %, `indistinguishable-from-baseline`). With all 23.5 M parameters free, 40 AdamW steps at learning rate 1e-4 on batches of 4 tiny, upscaled images move the pretrained backbone away from the features that made the task easy, and ResNet-50's BatchNorm layers re-estimate their running statistics from those 4-image batches, so the features the head sees at evaluation are no longer the pretrained ones. Head-only training keeps both fixed. The two intervals (71–93 % vs 38–67 %) do not overlap, so on this split head-only is clearly better. More parameters need more data or a gentler schedule: a smaller `LEARNING_RATE` (e.g. 1e-5) or a larger `BATCH_SIZE` is the natural next experiment for full fine-tuning. A GPU run can land a few images away from 21/40.</details>

In [ ]:
# Section 10 adds one row per fine-tuning run in this session; this cell only prints them.
columns = ['method', 'trainable_parameters', 'epochs', 'learning_rate', 'split_seed', 'final_val_loss', 'correct', 'n', 'wilson_95', 'comparison_to_baseline']
print(' | '.join(columns))
for row in run_history:
    print(' | '.join(str(row[column]) for column in columns))
if len(run_history) == 1:
    print("One run so far. Set TRAINABLE = 'all' in Section 9, select that cell and choose Runtime > Run after; this table then shows both runs.")

method | trainable_parameters | epochs | learning_rate | split_seed | final_val_loss | correct | n | wilson_95 | comparison_to_baseline
head-only fine-tuning | 4098 | 1 | 0.0001 | 42 | 0.6285 | 34 | 40 | [0.709, 0.929] | above-baseline
One run so far. Set TRAINABLE = 'all' in Section 9, select that cell and choose Runtime > Run after; this table then shows both runs.


## Interpretation and limits

*Evaluation practice.* The predicted label is the argmax of a softmax over the fixed 1000-class ImageNet-1k label space (or the custom class names after fine-tuning); the `score` values are uncalibrated softmax outputs, not probabilities of correctness, and the pipeline ships no threshold. On the synthetic gradient the label is meaningless by construction and the evaluation report says `not-measurable`; a `top_k_accuracy` value shown for a single BYOD image is tutorial evidence for that one image and must not be generalized to a domain, camera, or class distribution.

The **fine-tuning result** is a held-out accuracy on **40 images** — 20 photos (10 frogs, 10 trucks, 32×32 px upscaled), each with its darkened copy — from one seeded, pair-grouped split and one training run. Read it together with its 95 % interval and the 50 % majority-class baseline printed in Section 10, remembering that the interval treats the 40 correlated images as independent and is therefore too narrow. Different split seeds or devices move the count: a local CPU check of this version scored 34/40 at the default seed and 34/40, 37/40 and 32/40 at split seeds 0, 1 and 2 (head-only), while full fine-tuning (Section 12) fell to 21/40, no better than the baseline. It shows that the fine-tuning, export and reload path runs and produces a working two-class classifier on this sample; it does **not** show how well fine-tuning works on frogs and trucks, on CIFAR-10, or on your data. The default run trains only the 4,098-parameter head, so the frozen ImageNet features decide what the classifier can see; full fine-tuning of all 23.5 M parameters on 160 images is unstable with these settings, as Section 12 shows. Images whose subject is outside ImageNet-1k, line drawings, medical or satellite imagery, and subjects near the image border (removed by the center crop) all degrade results in ways the pipeline does not detect. The pipeline provides no detection, segmentation, multi-label, OCR, or open-vocabulary capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input, execute the public pipeline path, perform in-kernel fine-tuning, reload an artifact that reproduces the in-memory model, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments** (each starts after the default Run all):

1. **A photograph with a known class:** in Section 4 set `USE_BYOD = True` and `GROUND_TRUTH_INDEX` to its ImageNet-1k index, select the Section 4 cell and choose **Runtime → Run after**; Section 7 switches to `sample-sanity` with `top_k_accuracy` at k=1 and k=5. (Run after also repeats the fine-tuning in Sections 9–11.)
2. **Another split:** in Section 8 change `SPLIT_SEED`, then **Run after** from Section 8; Section 12 shows how the held-out count moves.
3. **Your own classes:** in Section 8 set `USE_BYOD_DATASET = True` (and upload, or set `BYOD_DATASET_PATH`), then **Run after** from Section 8.
4. **Gentler full fine-tuning:** after Section 12, keep `TRAINABLE = 'all'` and lower `LEARNING_RATE` (e.g. 1e-5) in Section 9, then **Run after** from Section 9; compare its row with the earlier full fine-tuning row.

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment is built from manylinux wheels; use Colab, Kaggle or a Linux Jupyter server.
- **Section 1 fails to download `uv`, Python or a package.** The runtime needs `pypi.org`, `files.pythonhosted.org` and the python-build-standalone release host. Re-run the cell; a size or SHA-256 mismatch is refused on purpose.
- **"The isolated environment's Python process exited".** Usually out of memory. Restart the session and choose **Run all** again; on a small CPU runtime lower `BATCH_SIZE` in Section 9.
- **Section 3 reports a size or SHA-256 mismatch.** A snapshot file was altered or truncated; delete `weights/resnet50-a1/model.safetensors` and run Section 3 again.
- **Section 8 prints "falling back to the deterministic synthetic stripes dataset".** `huggingface.co` was unreachable. The run continues on stripes; re-run Section 8 onward once the network is back. A SHA-256 mismatch of the downloaded archive stops the cell instead.
- **Section 8 refuses a BYOD archive.** The message names the file or class and the rule (class folders, matching `train/` and `val/` classes, no `test/` folder, at least 2 images per class, decodable images of at most 4096 px, the archive limits). Fix the archive and run Section 8 again.
- **Section 4 refuses a BYOD image** as not decodable: convert it to PNG or JPEG. Outside Colab there is no upload dialog; use `BYOD_IMAGE_PATH` / `BYOD_DATASET_PATH`.
- **Section 10 stops with "does not reproduce the in-memory model".** The export or reload is broken; re-run Section 9. Do not use that artifact.
- **CUDA out of memory.** Lower `BATCH_SIZE`, or switch the runtime to CPU.

## Glossary

- **ResNet-50:** a 50-layer convolutional network with residual (skip) connections (He et al., 2015); 25.6 M parameters with its 1000-class head. This checkpoint was trained on ImageNet-1k with the A1 recipe of *ResNet strikes back* (Wightman et al., 2021).
- **BatchNorm:** a layer that normalises features with a running mean and variance; in training mode it updates those statistics from each batch, in evaluation mode it uses the stored ones.
- **Logits / softmax:** the network's raw class scores; the softmax turns them into positive numbers summing to 1. A softmax score is a ranking signal, not a calibrated probability of being right.
- **Argmax / top-k:** the class with the highest score / the k highest-scoring classes in order.
- **Center crop:** after resizing the short side to 235 px, the central 224×224 square is kept; the border is discarded.
- **Head:** the final linear layer mapping features to class logits. **Backbone:** everything before it.
- **Full fine-tuning / head-only fine-tuning:** training every parameter / training only a new head with the backbone frozen.
- **AdamW:** the Adam optimiser with decoupled weight decay. **Cross-entropy:** the loss that penalises a low softmax score for the true class. **Epoch:** one pass over the training images.
- **Held-out (validation) set:** labelled images kept out of training and used only to evaluate.
- **Majority-class baseline:** the accuracy of always answering the most common class of the evaluated images.
- **95 % Wilson interval:** a range for the true accuracy that is consistent with the observed count at the 95 % level; with few images it is wide.
- **Fresh-boundary reload / equivalence:** loading the exported files into a new pipeline and checking that it predicts what the in-memory model predicted.

## Conclusion (your notes)

Fill in from the numbers this run printed; keep each claim to what the evidence shows.

- **Task:** what was classified, into which classes, from which data?
- **Principal result:** the held-out count, `n`, the 95 % interval, and the fine-tuning method.
- **Baseline / reference:** the majority-class baseline, and `comparison_to_baseline`.
- **Uncertainty or failure mode:** what would change with another split seed, another device, or more images? Which images were misclassified?
- **Limitations:** what does this run *not* show (see Interpretation and limits)?

## References

- Repository README: https://github.com/kurtvalcorza/resnet50-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/resnet50-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/resnet50-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/timm/resnet50.a1_in1k
- Upstream library: https://github.com/huggingface/pytorch-image-models
- ResNet strikes back (A1 training recipe): https://arxiv.org/abs/2110.00476
- Deep Residual Learning for Image Recognition: https://arxiv.org/abs/1512.03385- Tutorial dataset: https://huggingface.co/datasets/Cleanlab/cifar-10-subset
- Wilson, E. B. (1927), Probable inference, the law of succession, and statistical inference, JASA 22(158)